In [1]:
import sys  
sys.path.insert(1, '/home/spuchin/GitHub/the-hand-of-midas')

from clickhouse_connect.driver.asyncclient import AsyncClient as AsyncClickHouseClient
from pandas import DataFrame
import plotly.graph_objects as go
from talib._ta_lib import EMA

from src.adapters.repositories.candlesticks import CandlesticksRepository
from src.adapters.repositories.common.clickhouse_base import get_clickhouse_client
from src.schemas.binance import BinanceIntervalEnum, BinanceSectionEnum
from src.schemas.candlesticks import CandlesticksInputSchema
from src.services.candlesticks import CandlesticksService
from src.settings import settings

In [6]:
clickhouse_client: AsyncClickHouseClient = await get_clickhouse_client()
candlesticks_repository: CandlesticksRepository = CandlesticksRepository(client=clickhouse_client)
candlesticks_service: CandlesticksService = CandlesticksService(repository=candlesticks_repository)

ohlc: DataFrame = await candlesticks_service.get_candlesticks(
    input_schema=CandlesticksInputSchema(
        ticker=settings.TICKER,
        exchange=settings.BINANCE_EXCHANGE_NAME,
        section=BinanceSectionEnum.BINANCE_SPOT,
        interval=BinanceIntervalEnum.FOUR_HOURS
    )
)
ohlc.sort_values(by="datetime", inplace=True)
ohlc.tail(1)

,open,high,low,close,datetime
52,107198.02,108135.3,106927.8,107353.08,2025-06-25 12:00:00


### DS-3: [feature] Add macro candles.

In [7]:
MACRO_EXPONENTIONAL_MOVING_AVERAGES: int = 2 ** 8

ohlc["open_macro"] = EMA(ohlc.open.values, MACRO_EXPONENTIONAL_MOVING_AVERAGES)
ohlc["high_macro"] = EMA(ohlc.high.values, MACRO_EXPONENTIONAL_MOVING_AVERAGES)
ohlc["low_macro"] = EMA(ohlc.low.values, MACRO_EXPONENTIONAL_MOVING_AVERAGES)
ohlc["close_macro"] = EMA(ohlc.close.values, MACRO_EXPONENTIONAL_MOVING_AVERAGES)

ohlc["open_macro"] = ohlc["open_macro"].shift(1)
ohlc["high_macro"] = ohlc["high_macro"].shift(1)
ohlc["low_macro"] = ohlc["low_macro"].shift(1)
ohlc["close_macro"] = ohlc["close_macro"].shift(1)

### DS-10: [feature] Add micro candles.

In [8]:
MICRO_EXPONENTIONAL_MOVING_AVERAGES: int = 2 ** 4

ohlc["open_micro"] = EMA(ohlc.open.values, MICRO_EXPONENTIONAL_MOVING_AVERAGES)
ohlc["high_micro"] = EMA(ohlc.high.values, MICRO_EXPONENTIONAL_MOVING_AVERAGES)
ohlc["low_micro"] = EMA(ohlc.low.values, MICRO_EXPONENTIONAL_MOVING_AVERAGES)
ohlc["close_micro"] = EMA(ohlc.close.values, MICRO_EXPONENTIONAL_MOVING_AVERAGES)

ohlc["open_micro"] = ohlc["open_micro"].shift(1)
ohlc["high_micro"] = ohlc["high_micro"].shift(1)
ohlc["low_micro"] = ohlc["low_micro"].shift(1)
ohlc["close_micro"] = ohlc["close_micro"].shift(1)

### DS-12: [ad-hoc] Add market structure visualiztion for macro and micro candles.

In [9]:
number_of_rows: int = 1000

macro_candlesticks = go.Candlestick(
    x=ohlc["datetime"].tail(number_of_rows),
    open=ohlc["open_macro"].tail(number_of_rows),
    high=ohlc["high_macro"].tail(number_of_rows),
    low=ohlc["low_macro"].tail(number_of_rows),
    close=ohlc["close_macro"].tail(number_of_rows),
    showlegend=False
)
micro_candlesticks = go.Candlestick(
    x=ohlc["datetime"].tail(number_of_rows),
    open=ohlc["open_micro"].tail(number_of_rows),
    high=ohlc["high_micro"].tail(number_of_rows),
    low=ohlc["low_micro"].tail(number_of_rows),
    close=ohlc["close_micro"].tail(number_of_rows),
    increasing_line_color='cyan', decreasing_line_color='blue',
    showlegend=False
)
noise_candlesticks = go.Candlestick(
    x=ohlc["datetime"].tail(number_of_rows),
    open=ohlc["open"].tail(number_of_rows),
    high=ohlc["high"].tail(number_of_rows),
    low=ohlc["low"].tail(number_of_rows),
    close=ohlc["close"].tail(number_of_rows),
    increasing_line_color='gray', decreasing_line_color='black',
    showlegend=False
)

figure = go.Figure(data=[noise_candlesticks, micro_candlesticks, macro_candlesticks])

figure.update_layout(xaxis_rangeslider_visible=False)
figure.show()

In [10]:
ohlc.to_csv("macro-micro-candles.csv", index=False)

---